# Topic I — Numerical Integration & Root-Finding

*FINM 32000 Numerical Methods — Final-Exam Study Notes*

---

**Prerequisites:** [Topic 0 — Foundations](Topic-0_Foundations.ipynb) (risk-neutral pricing, BS), [Topic E — MC](Topic-E_L5_MonteCarlo_Foundations.ipynb).

**What this notebook covers**
1. Where numerical integration shows up in option pricing
2. **Quadrature rules** — midpoint, trapezoidal, Simpson (and their error orders)
3. **Practical Python tools** — `scipy.integrate.quad`, `numpy.trapezoid`
4. **Worked example 1** — compute $\int_0^T \sigma^2(t)\,dt$ for time-dependent vol (Topic B)
5. **Worked example 2** — price a call via "payoff × density" integration (L8.35)
6. **Root-finding for implied volatility** — Brent's method via `scipy.optimize.brentq`
7. When to use quadrature vs MC vs FD
8. Cheat sheet

**Exam map**

| Source | What it tests |
|---|---|
| L8.35 | One-dimensional integration of payoff × density |
| L8.36 | Numerical integration in Fourier space (mentioned, not on final) |
| HW1, HW5 P2 | `brentq` for implied vol inversion |
| Topic B (L2.29) | Integrating piecewise volatility |
| Practice-Final P2 | Same |

**Note on exam weight:** Numerical integration is a **light topic** on this exam. The professor said most of L8 is **excluded** (only L8.25–L8.30 Heston skews possible — that is Topic J). Use this notebook as a **toolbox reference** more than a deep dive.


In [1]:
import numpy as np
from scipy.stats import norm
from scipy.integrate import quad, simpson
from scipy.optimize import brentq
import matplotlib.pyplot as plt
%matplotlib inline
np.set_printoptions(precision=8, suppress=True)

def bs_call(S, K, T, r, sigma, q=0.0):
    if T <= 0:
        return max(S - K, 0.0)
    std = sigma * np.sqrt(T)
    d1 = (np.log(S/K) + (r - q + 0.5*sigma**2)*T) / std
    d2 = d1 - std
    return S*np.exp(-q*T)*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)


## 1. Where does numerical integration appear in this course?

| Where | Integral | Why we need numerical methods |
|---|---|---|
| **Time-dependent vol** (Topic B) | $\bar\sigma^2 T = \int_0^T \sigma^2(t)\,dt$ | Closed-form only for special $\sigma(t)$; quadrature for general functions |
| **Implied vol inversion** (HW1) | Solve $C_{BS}(\sigma) = V_{\text{market}}$ for $\sigma$ | Root-finding (Brent) |
| **Density × payoff** (L8.35) | $C = \int (e^x - K)^+ f(x)\,dx$ | When you have a density (not a CF) |
| **MC** (Topic E) | $\mathbb E[Y]$ | Random sampling = stochastic integration |
| **FD** (Topic C/D) | Discretized PDE | Implicit deterministic integration |
| **Fourier** (L8) | $C = \frac{1}{\pi}\int \mathrm{Re}[\dots]\,dz$ | Heston, mostly **excluded** from this final |

The main **deterministic** integration tools are quadrature rules and root-finders. We cover them here.


## 2. Quadrature rules

The setup: compute $\int_a^b f(x)\,dx$ given the ability to evaluate $f$ at chosen points. Subdivide into $N$ subintervals of width $h = (b-a)/N$, with nodes $x_n = a + n h$.

### 2.1 Midpoint rule (L8.35)

$$\int_a^b f(x)\,dx \approx h \sum_{n=1}^N f\bigl(a + (n - \tfrac12)h\bigr).$$

**Plain English:** Evaluate $f$ at the **center** of each interval, multiply by width, sum.

**Error per interval:** $O(h^3)$ (one Taylor term cancels by symmetry).  
**Total error:** $N \cdot O(h^3) = O(h^2)$.

### 2.2 Trapezoidal rule

$$\int_a^b f(x)\,dx \approx \frac{h}{2}\Bigl[f(x_0) + 2\sum_{n=1}^{N-1} f(x_n) + f(x_N)\Bigr].$$

**Plain English:** Approximate $f$ as a straight line between each pair of nodes; integrate the trapezoids.

**Total error:** $O(h^2)$ — same order as midpoint, but typically worse by a factor of 2 for smooth $f$.

### 2.3 Simpson's rule (need $N$ even)

$$\int_a^b f(x)\,dx \approx \frac{h}{3}\Bigl[f(x_0) + 4\sum_{n\text{ odd}} f(x_n) + 2\sum_{n\text{ even}, n\ne 0,N} f(x_n) + f(x_N)\Bigr].$$

**Plain English:** Fit a parabola through every three consecutive nodes; integrate.

**Total error:** $O(h^4)$ — much better than midpoint/trapezoidal for smooth $f$.

### Error comparison

| Rule | Per-interval | Total over $[a,b]$ | When best |
|---|---|---|---|
| Midpoint | $O(h^3)$ | $O(h^2)$ | $f$ undefined at endpoints |
| Trapezoidal | $O(h^3)$ | $O(h^2)$ | Quick & simple |
| **Simpson's** | $O(h^5)$ | $O(h^4)$ | $f$ is smooth (4 derivatives) |
| Adaptive (`quad`) | varies | machine precision | General-purpose; uses Gauss-Kronrod |

### Python tools

| Goal | Code |
|---|---|
| Adaptive — best for one-off computations | `scipy.integrate.quad(f, a, b)` |
| Fixed-grid Simpson | `scipy.integrate.simpson(y, x=x)` |
| Fixed-grid trapezoidal | `np.trapz(y, x=x)` (NumPy 2.0+: `np.trapezoid`) |
| Custom midpoint | one-line numpy `np.mean(f(x_mid)) * (b-a)` |


In [2]:
# Convergence comparison for a smooth function: integral_0^1 cos(2*pi*x) dx = 0
def f(x):
    return np.cos(2*np.pi*x) * np.exp(-x)

# True value (computed by quad)
truth, _ = quad(f, 0, 1)
print(f"True value of integral: {truth:.10f}\n")

print(f"{'N':>5} | {'midpoint err':>15} | {'trapezoidal err':>17} | {'Simpson err':>15}")
print("-" * 65)
for N in [4, 8, 16, 32, 64, 128, 256]:
    h = 1.0 / N

    # Midpoint
    x_mid = np.linspace(h/2, 1 - h/2, N)
    mid = h * f(x_mid).sum()

    # Trapezoidal
    x = np.linspace(0, 1, N + 1)
    trap = np.trapz(f(x), x=x)

    # Simpson (N must be even)
    simp = simpson(f(x), x=x)

    print(f"{N:>5} | {abs(mid-truth):15.2e} | {abs(trap-truth):17.2e} | {abs(simp-truth):15.2e}")

print("\nNote: midpoint/trapezoidal error halves 4x when N doubles (O(h^2)).")
print("      Simpson error shrinks 16x when N doubles (O(h^4)).")


True value of integral: 0.0156162369

    N |    midpoint err |   trapezoidal err |     Simpson err
-----------------------------------------------------------------
    4 |        2.04e-03 |          3.74e-03 |        2.71e-03
    8 |        4.34e-04 |          8.49e-04 |        1.14e-04
   16 |        1.04e-04 |          2.07e-04 |        6.48e-06
   32 |        2.58e-05 |          5.15e-05 |        3.96e-07
   64 |        6.44e-06 |          1.29e-05 |        2.46e-08
  128 |        1.61e-06 |          3.22e-06 |        1.54e-09
  256 |        4.02e-07 |          8.04e-07 |        9.60e-11

Note: midpoint/trapezoidal error halves 4x when N doubles (O(h^2)).
      Simpson error shrinks 16x when N doubles (O(h^4)).


## 3. Worked example 1 — integrated variance for time-dependent vol

### Setup (Topic B, L2.29)

For $dS_t = \sigma(t) S_t\,dW_t$ with deterministic $\sigma(t)$, the time-0 implied vol for a $T$-expiry option is
$$\bar\sigma_T := \sqrt{\frac{1}{T}\int_0^T \sigma^2(t)\,dt}.$$

**For piecewise-constant $\sigma$** (Topic B, HW2), the integral is exact by hand. **For general $\sigma(t)$** you need quadrature.

### Example: $\sigma(t) = 0.20 + 0.15\sin(2\pi t)$ on $[0, T=1]$

Compute $\bar\sigma_T^2$ three ways.


In [3]:
def sigma_t(t):
    return 0.20 + 0.15 * np.sin(2 * np.pi * t)

def integrand(t):
    return sigma_t(t)**2

T = 1.0

# Method 1: scipy.integrate.quad (adaptive Gauss-Kronrod, near machine precision)
exact, err = quad(integrand, 0, T)
print(f"Method 1 — adaptive quad: integral = {exact:.10f}  (estimated error {err:.2e})")
print(f"            sigma_bar = {np.sqrt(exact/T):.6f}")

# Method 2: Simpson on a uniform grid
N = 100
t = np.linspace(0, T, N + 1)
simp = simpson(integrand(t), x=t)
print(f"\nMethod 2 — Simpson N=100:  integral = {simp:.10f}")
print(f"            sigma_bar = {np.sqrt(simp/T):.6f}")

# Method 3: trapezoidal (less accurate)
trap = np.trapz(integrand(t), x=t)
print(f"\nMethod 3 — Trapezoid N=100: integral = {trap:.10f}")
print(f"            sigma_bar = {np.sqrt(trap/T):.6f}")

# Closed-form for sanity:  E[sigma^2] over [0,1] for this specific sigma
# integral_0^1 (a + b*sin(2*pi t))^2 dt = a^2 + b^2/2  (cross term integrates to 0)
closed = 0.20**2 + 0.15**2 / 2
print(f"\nAnalytic answer (this sigma):   integral = {closed:.10f}")


Method 1 — adaptive quad: integral = 0.0512500000  (estimated error 3.57e-12)
            sigma_bar = 0.226385

Method 2 — Simpson N=100:  integral = 0.0512500000
            sigma_bar = 0.226385

Method 3 — Trapezoid N=100: integral = 0.0512500000
            sigma_bar = 0.226385

Analytic answer (this sigma):   integral = 0.0512500000


## 4. Worked example 2 — call price as "payoff × density" (L8.35)

For an asset $S_T = e^{X_T}$ with **known density** $f_X$ of $\log S_T$, a European call has price
$$C = e^{-rT}\,\mathbb E[(S_T - K)^+] = e^{-rT}\int_{\log K}^{\infty} (e^x - K)\,f_X(x)\,dx.$$

For GBM, $X_T \sim \mathcal N(\log S_0 + (r - \tfrac12\sigma^2)T,\, \sigma^2 T)$ — a Gaussian density. We can integrate numerically and check against Black-Scholes.

### Why bother when BS has a closed form?

For local-vol or jump models, **$f_X$** may be available (or computable from a CF via inversion) but **no closed-form price exists**. Numerical integration then is the cheapest pricing route.


In [4]:
S0, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.20
mu_X = np.log(S0) + (r - 0.5*sigma**2)*T
sd_X = sigma * np.sqrt(T)

def f_X(x):
    return norm.pdf(x, loc=mu_X, scale=sd_X)

def integrand(x):
    return (np.exp(x) - K) * f_X(x)

# Compute the integral from log(K) to "infinity" — use a generous upper bound
# Cut off at mu + 10 std deviations: density is ~1e-22 there, contributing negligibly
upper = mu_X + 10 * sd_X
integral, err = quad(integrand, np.log(K), upper)
price_quad = np.exp(-r*T) * integral

price_bs = bs_call(S0, K, T, r, sigma)

print(f"Quadrature price:    {price_quad:.6f}")
print(f"Black-Scholes price: {price_bs:.6f}")
print(f"Difference:          {abs(price_quad - price_bs):.2e}")


Quadrature price:    10.450584
Black-Scholes price: 10.450584
Difference:          4.62e-14


## 5. Root-finding for implied volatility

### The problem

Given a market call price $V_{\text{market}}$, find $\sigma$ such that
$$C_{BS}(S_0, K, T, r, \sigma) = V_{\text{market}}.$$

$C_{BS}$ is **monotonically increasing in $\sigma$** (vega is positive), so the solution is unique.

### Brent's method (`scipy.optimize.brentq`)

Brent's method is a robust hybrid of:
- **Bisection** — guaranteed to converge if you bracket the root with a sign change
- **Secant / inverse quadratic interpolation** — fast superlinear convergence

You provide:
- A function `f(sigma) = C_BS(sigma) - V_market`
- A bracket `[sigma_lo, sigma_hi]` with opposite signs

`brentq` finds the root in typically 10–20 iterations to machine precision.

### Why not Newton's method?

Newton's $\sigma_{n+1} = \sigma_n - \frac{f(\sigma_n)}{\text{vega}(\sigma_n)}$ is faster (quadratic convergence) but
- can diverge if you start far from the answer (deep OTM with tiny vega), and
- has no global guarantee.

For implied vol on the final exam, `brentq` is the standard.

### Code template

```python
from scipy.optimize import brentq
def implied_vol(V_market, S, K, T, r):
    return brentq(lambda sig: bs_call(S, K, T, r, sig) - V_market, 1e-6, 5.0)
```

The bracket $[10^{-6}, 5]$ covers virtually all realistic vols. A tighter bracket converges marginally faster.


In [5]:
def implied_vol(V_market, S, K, T, r, q=0.0):
    return brentq(lambda sig: bs_call(S, K, T, r, sig, q) - V_market, 1e-8, 5.0)

# Generate a market price from sigma = 0.25, then recover sigma
S, K, T, r, sigma_true = 100, 100, 0.5, 0.03, 0.25
V_market = bs_call(S, K, T, r, sigma_true)
sigma_hat = implied_vol(V_market, S, K, T, r)

print(f"True sigma:      {sigma_true:.6f}")
print(f"Recovered sigma: {sigma_hat:.6f}")
print(f"Difference:      {abs(sigma_hat - sigma_true):.2e}")

# Now do a small market: multiple strikes, recover the implied vol curve
strikes = np.array([80, 90, 100, 110, 120])
# Pretend the true vols form a smile
true_vols = np.array([0.25 + 0.02*(np.log(K/100))**2 for K in strikes])
mkt_prices = np.array([bs_call(S, K, T, r, sig) for K, sig in zip(strikes, true_vols)])
recovered = np.array([implied_vol(V, S, K, T, r) for K, V in zip(strikes, mkt_prices)])

print(f"\n{'Strike':>8} | {'Price':>8} | {'True IV':>8} | {'IV':>8}")
print("-" * 45)
for K, p, t_iv, iv in zip(strikes, mkt_prices, true_vols, recovered):
    print(f"{K:>8.0f} | {p:>8.4f} | {t_iv:>8.4f} | {iv:>8.4f}")


True sigma:      0.250000
Recovered sigma: 0.250000
Difference:      2.78e-16

  Strike |    Price |  True IV |       IV
---------------------------------------------
      80 |  21.8451 |   0.2510 |   0.2510
      90 |  13.7955 |   0.2502 |   0.2502
     100 |   7.7603 |   0.2500 |   0.2500
     110 |   3.9033 |   0.2502 |   0.2502
     120 |   1.7799 |   0.2507 |   0.2507


## 6. When to use what — a decision flowchart

```mermaid
flowchart TD
    A[I need to compute a number] --> B{Random or deterministic?}
    B -->|Random expectation| C{Closed form?}
    C -->|Yes| D[Use formula]
    C -->|No, low-dimensional| E[Quadrature: density x payoff]
    C -->|No, high-dimensional| F[Monte Carlo - Topic E/F]
    B -->|Deterministic PDE| G[Finite differences - Topic C/D]
    B -->|Deterministic integral 1D| H{Smooth integrand?}
    H -->|Yes| I[scipy.integrate.quad]
    H -->|Singular or oscillatory| J[Custom quadrature / Fourier]
    A --> K{Solve f x equals c?}
    K -->|Monotone, bracketable| L[brentq - Brent's method]
    K -->|Multivariate / nonlinear| M[scipy.optimize]
```

### Rules of thumb

- **For 1-D smooth integrals**: `quad` is overkill-but-safe and the default.
- **For an integral on a fixed grid you already have**: `simpson` (best accuracy) or `np.trapz` (simpler).
- **For implied vol or any monotone inverse**: `brentq`.
- **For high-dimensional integrals** ($D \ge 4$): MC beats quadrature (quadrature scales as $N^D$).


## 7. Common pitfalls

### Pitfall 1 — using a small upper limit on infinite integrals

If you integrate $\int_a^\infty$ via quadrature, replace $\infty$ with a generous cutoff (e.g., $\mu + 10\sigma$ for Gaussian-like densities). Too small a cutoff truncates the tail.

### Pitfall 2 — Simpson's rule needs $N$ even

If you pass an odd number of intervals to Simpson's rule, behavior depends on the library. `scipy.integrate.simpson` will warn/use trapezoid for the leftover.

### Pitfall 3 — `brentq` requires a sign change in the bracket

If $V_{\text{market}}$ is less than the **lower bound** of BS call prices (e.g., negative price, or below intrinsic), `brentq` fails. Check feasibility first.

### Pitfall 4 — Brent's method is for **1-variable** problems

For $K$ unknowns, use `scipy.optimize.root` (multidimensional). For local-vol calibration etc.

### Pitfall 5 — vega vanishes far from the money

For deep-OTM calls, $C_{BS}(\sigma)$ is nearly flat in $\sigma$: many values produce the same price. The implied vol is sensitive to small price errors. Use **OTM puts** for ITM strikes (via put-call parity) — they have larger absolute vega.

### Pitfall 6 — integration accuracy is wasted if your **integrand has noise**

If the integrand itself comes from MC, no amount of clever quadrature fixes the noise. Reduce $\hat\sigma_M$ first.


## 8. Exam-style exercises

**E1.** Why does Simpson's rule beat the trapezoidal rule by an order of $h^2$ for smooth integrands?

**E2.** A call has $V_{\text{market}} = 3.50$, $S = 100$, $K = 100$, $T = 0.25$, $r = 0.05$. Estimate the implied vol to 3 decimals.

**E3.** For $\sigma(t) = 0.20$ on $[0, 0.1]$ and $\sigma(t) = 0.40$ on $[0.1, 0.3]$, compute $\bar\sigma_{0.3}$ exactly (no quadrature needed).

**E4.** Why does midpoint quadrature have error $O(h^2)$ even though it uses only one function evaluation per interval?


### Solutions

<details><summary><b>E1</b></summary>

Simpson fits a parabola (degree 2 polynomial) through 3 nodes; the error involves the 4th derivative ($O(h^4)$). Trapezoidal fits a line (degree 1); error involves 2nd derivative ($O(h^2)$). Two extra cancellations come from symmetry around the midpoint.
</details>

<details><summary><b>E2</b></summary>

Use `brentq` on $f(\sigma) = C_{BS}(\sigma) - 3.50$. The answer is approximately $\sigma \approx 0.349$.
</details>

<details><summary><b>E3</b></summary>

$\int_0^{0.3}\sigma^2(t)\,dt = 0.1 \cdot 0.20^2 + 0.2 \cdot 0.40^2 = 0.004 + 0.032 = 0.036$. So $\bar\sigma_{0.3} = \sqrt{0.036/0.3} = \sqrt{0.12} \approx 0.3464$.
</details>

<details><summary><b>E4</b></summary>

The leading-order error term is $-\frac{f''(x_n^*)}{24}h^3$ per interval (Taylor at midpoint; first derivative cancels by symmetry). Times $N = (b-a)/h$ intervals gives $O(h^2)$ total.
</details>


## 9. Cheat sheet — Topic I

| # | Tool | When | Order |
|---|---|---|---|
| 1 | `scipy.integrate.quad` | 1-D smooth, one-off | adaptive (~machine precision) |
| 2 | `scipy.integrate.simpson` | 1-D on a fixed grid | $O(h^4)$ |
| 3 | `np.trapz` | 1-D, simple | $O(h^2)$ |
| 4 | Midpoint rule | $f$ undefined at boundaries | $O(h^2)$ |
| 5 | `scipy.optimize.brentq` | 1-D root of monotone $f$ | guaranteed, ~superlinear |
| 6 | MC | high-dim expectations | $O(1/\sqrt M)$ — see Topic E |
| 7 | FD | PDE / Greeks on grid | $O(\Delta t) + O(\Delta x^2)$ explicit |
| 8 | Implied vol bracket | $[10^{-8}, 5]$ | almost always works |

### Black-Scholes call as "density × payoff"

For GBM, $\log S_T \sim \mathcal N(\log S_0 + (r-\tfrac12\sigma^2)T,\,\sigma^2 T)$, so
$$C = e^{-rT}\int_{\log K}^{\infty}(e^x - K)\,\frac{1}{\sigma\sqrt{2\pi T}}\exp\!\Bigl(-\frac{(x-\mu)^2}{2\sigma^2 T}\Bigr)dx.$$

For non-GBM models with known density, replace the Gaussian with the appropriate $f_X$.

### Implied vol template

```python
from scipy.optimize import brentq

def implied_vol(V, S, K, T, r):
    return brentq(lambda sig: bs_call(S, K, T, r, sig) - V, 1e-8, 5.0)
```

---

*Skipped (for now):* **Topic G** (MC delta bias-variance — Practice-Final P4) and **Topic H** (Longstaff-Schwartz — Practice-Final P5). Both are exam-relevant; ask to revisit them when you're ready.

*Next planned:* **Topic J** — Heston model skews (L8.25–L8.30), if the professor includes it.
